# 13 · Organization-Wide Skill Gap Rollup & Concentration Analysis

**Project:** Enterprise HR AI  

> ### ⚠️ PROMINENT DATA INTEGRITY WARNING
> **SYNTHETIC DATA — employee current-skill possession was not present in any source file and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. This must NOT be presented to stakeholders as real observed skill data. Real deployment requires an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).**

---

---
## Step 1 · Load Employee Skill Gaps & Explode Missing Skills

Loading `data/processed/employee_skill_gaps.csv` (1,368 non-manager employees) and exploding semicolon-separated `missing_skills` into individual `(EmployeeNumber, JobRole, skill_name)` tuples.

In [1]:
import pandas as pd
import numpy as np
import os

PROC = os.path.join('..', 'data', 'processed')
gaps_path = os.path.join(PROC, 'employee_skill_gaps.csv')

# Load employee skill gaps skipping the header comment
df_gaps = pd.read_csv(gaps_path, comment='#')
print(f'Loaded employee gap profiles: {len(df_gaps):,} employees')

# Explode missing_skills
exploded_records = []
for _, row in df_gaps.iterrows():
    if pd.isna(row['missing_skills']):
        continue
    missing_str = str(row['missing_skills']).strip()
    if missing_str in ('', 'None', 'nan'):
        continue
        
    skills = [s.strip() for s in missing_str.split(';') if s.strip()]
    for s in skills:
        exploded_records.append({
            'EmployeeNumber': row['EmployeeNumber'],
            'JobRole': row['JobRole'],
            'skill_name': s
        })

df_exploded = pd.DataFrame(exploded_records)
print(f'Total exploded (employee, missing_skill) pairs: {len(df_exploded):,}')
print(f'Unique skills represented across all gaps     : {df_exploded["skill_name"].nunique():,}')

Loaded employee gap profiles: 1,368 employees
Total exploded (employee, missing_skill) pairs: 3,785
Unique skills represented across all gaps     : 33


---
## Step 2 · Roll Up Organization-Wide & Apply Severity Rule

**Explicit Severity Threshold Rule:**
- **HIGH Severity:** $\ge 100$ employees missing the skill across the organization
- **MEDIUM Severity:** $50$ to $99$ employees missing the skill
- **LOW Severity:** $< 50$ employees missing the skill

In [2]:
# Group and count total missing employees per skill
skill_summary = (
    df_exploded.groupby('skill_name')
    .size()
    .reset_index(name='total_missing_count')
    .sort_values('total_missing_count', ascending=False)
    .reset_index(drop=True)
)

# Apply explicit severity rule
def classify_severity(count):
    if count >= 100:
        return 'HIGH'
    elif count >= 50:
        return 'MEDIUM'
    else:
        return 'LOW'

skill_summary['severity'] = skill_summary['total_missing_count'].apply(classify_severity)

# Print severity band breakdown
sev_dist = skill_summary['severity'].value_counts()[['HIGH', 'MEDIUM', 'LOW']]
print('=== SEVERITY BAND DISTRIBUTION ===')
for band, count in sev_dist.items():
    print(f'  {band:<8}: {count:2d} skills ({count/len(skill_summary)*100:.1f}%)')
print(f'  Total   : {len(skill_summary):2d} unique skills\n')

print('=== FULL RANKED ORGANIZATION SKILL GAP INVENTORY (33 skills) ===')
print(f'{"Rank":<5} {"Skill Name":<42} {"Missing Count":<15} {"Severity":<10}')
print('-' * 75)
for idx, r in skill_summary.iterrows():
    print(f'{idx+1:<5} {r["skill_name"]:42} {r["total_missing_count"]:>13}   {r["severity"]:10}')

=== SEVERITY BAND DISTRIBUTION ===
  HIGH    : 11 skills (33.3%)
  MEDIUM  : 16 skills (48.5%)
  LOW     :  6 skills (18.2%)
  Total   : 33 unique skills

=== FULL RANKED ORGANIZATION SKILL GAP INVENTORY (33 skills) ===
Rank  Skill Name                                 Missing Count   Severity  
---------------------------------------------------------------------------
1     Speaking                                             388   HIGH      
2     Reading Comprehension                                378   HIGH      
3     Active Listening                                     375   HIGH      
4     Critical Thinking                                    307   HIGH      
5     Microsoft Office software                            153   HIGH      
6     Active Learning                                      151   HIGH      
7     Adobe Acrobat                                        151   HIGH      
8     Microsoft Excel                                      144   HIGH      
9     Monitoring    

---
## Step 3 · Cross-Reference Top Affected Roles & Concentration Flag

**Role-Concentration Heuristic:**  
A skill is flagged as **Role-Concentrated** (`is_role_concentrated = True`) if **$\ge 80\%$** of its organization-wide missing count originates from a **single job role**.  
Otherwise, it is designated as **Cross-Cutting** (`is_role_concentrated = False`).

> **Training Budget & Strategic Implications:**
> - **Cross-Cutting Skills** (e.g., *Speaking, Reading Comprehension, Critical Thinking, MS Office, Excel*): Warrant company-wide L&D initiatives, centralized asynchronous learning platforms, or general onboarding modules.
> - **Role-Concentrated Skills** (e.g., *AWS CloudFormation, EC2, DynamoDB, MEDITECH, AutoCAD*): Require targeted departmental budget allocation — funding specialized external certifications or domain bootcamps for specific teams rather than diluting funds across broad enterprise training.

In [3]:
CONCENTRATION_THRESHOLD = 0.80

top_roles_list = []
is_conc_list = []
max_role_pct_list = []

for _, row in skill_summary.iterrows():
    s_name = row['skill_name']
    sub = df_exploded[df_exploded['skill_name'] == s_name]
    role_counts = sub['JobRole'].value_counts()
    
    # Format role string
    role_str = ', '.join([f'{role} ({cnt})' for role, cnt in role_counts.items()])
    top_roles_list.append(role_str)
    
    # Max share
    top_share = role_counts.iloc[0] / len(sub)
    max_role_pct_list.append(round(top_share * 100, 1))
    is_conc_list.append(top_share >= CONCENTRATION_THRESHOLD)

skill_summary['top_affected_roles'] = top_roles_list
skill_summary['is_role_concentrated'] = is_conc_list
skill_summary['max_role_share_pct'] = max_role_pct_list

print('=== TOP 10 ORGANIZATION SKILL GAPS WITH ROLE BREAKDOWN ===\n')
for idx in range(10):
    r = skill_summary.iloc[idx]
    flag = 'ROLE-CONCENTRATED (Targeted)' if r['is_role_concentrated'] else 'CROSS-CUTTING (Company-Wide)'
    print(f'{idx+1:2d}. {r["skill_name"]} (Total Missing: {r["total_missing_count"]}) — [{r["severity"]}] — {flag}')
    print(f'    Max Role Share : {r["max_role_share_pct"]}%')
    print(f'    Role Breakdown : {r["top_affected_roles"]}\n')

# Concentration summary
conc_counts = skill_summary['is_role_concentrated'].value_counts()
print('=== CONCENTRATION DISTRIBUTION OVER ALL 33 SKILLS ===')
print(f'  Role-Concentrated (>=80% from 1 role) : {conc_counts.get(True, 0)} skills')
print(f'  Cross-Cutting (<80% concentration)    : {conc_counts.get(False, 0)} skills')

=== TOP 10 ORGANIZATION SKILL GAPS WITH ROLE BREAKDOWN ===

 1. Speaking (Total Missing: 388) — [HIGH] — CROSS-CUTTING (Company-Wide)
    Max Role Share : 26.3%
    Role Breakdown : Research Scientist (102), Sales Executive (90), Laboratory Technician (83), Manufacturing Director (36), Healthcare Representative (33), Sales Representative (26), Human Resources (18)

 2. Reading Comprehension (Total Missing: 378) — [HIGH] — CROSS-CUTTING (Company-Wide)
    Max Role Share : 22.5%
    Role Breakdown : Sales Executive (85), Research Scientist (84), Laboratory Technician (80), Healthcare Representative (32), Sales Representative (31), Manufacturing Director (29), Research Director (21), Human Resources (16)

 3. Active Listening (Total Missing: 375) — [HIGH] — CROSS-CUTTING (Company-Wide)
    Max Role Share : 24.3%
    Role Breakdown : Research Scientist (91), Laboratory Technician (79), Sales Executive (71), Healthcare Representative (34), Manufacturing Director (33), Sales Representative (

---
## Step 4 · Save Output Dataset (`organization_skill_gaps.csv`)

Saving the organization-wide gap inventory to `data/processed/organization_skill_gaps.csv`.  
The synthetic warning comment is retained in line 1.

In [4]:
out_file = os.path.join(PROC, 'organization_skill_gaps.csv')

# Export columns requested
export_cols = ['skill_name', 'total_missing_count', 'severity', 'top_affected_roles', 'is_role_concentrated']
df_export = skill_summary[export_cols]

warning_comment = (
    '# SYNTHETIC DATA — employee current-skill possession was not present in any source file '
    'and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. '
    'This must NOT be presented to stakeholders as real observed skill data. Real deployment requires '
    'an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).\n'
)

with open(out_file, 'w', encoding='utf-8') as f:
    f.write(warning_comment)
    df_export.to_csv(f, index=False)

file_size = os.path.getsize(out_file)
print(f'Saved organization skill gaps to: {out_file}')
print(f'File size: {file_size:,} bytes')
print(f'Total skills recorded: {len(df_export)}')

# Round-trip reload verification
df_reloaded = pd.read_csv(out_file, comment='#')
assert len(df_reloaded) == 33, 'Row count mismatch on reload!'
assert list(df_reloaded.columns) == export_cols, 'Column mismatch on reload!'
print('CONFIRMED: Round-trip verification passed cleanly.')

Saved organization skill gaps to: ..\data\processed\organization_skill_gaps.csv
File size: 3,696 bytes
Total skills recorded: 33
CONFIRMED: Round-trip verification passed cleanly.
